# T1.2 — The leakage-free baseline

**Build-Instructions task:** T1.2 · **Spec:** `TRD.md §2.2`, `PRD.md §3` Contribution 1

## What this notebook is

`01_reproduce_leakage.ipynb` showed the base paper's pipeline order is broken. This notebook
rebuilds the same pipeline with **one thing changed** — the split moved before the resampling —
and reports the resulting honest numbers.

> **The numbers in this notebook are the ones this project cites everywhere else.** The T1.1
> figures exist only as evidence of a defect; they are never a result and never a target.

## The corrected pipeline (`TRD.md §2.2`)

```
clean → label-encode → feature-select (TRAIN FOLD ONLY)
      → SPLIT 80/10/10
      → SMOTE (TRAIN FOLD ONLY)
      → MinMax scale (scaler FITTED ON TRAIN FOLD ONLY)
      → train → evaluate
```

Three things move, not one. Beyond the SMOTE ordering, **feature selection** and the **scaler**
are also fitted on the training fold only. Fitting either on the full dataset leaks test-fold
information into the model — a milder version of the same bug, and one the base paper's described
order commits as well.

Everything else — the cleaning, the 62-feature budget, the classifier and its hyperparameters,
the seed — is held identical to T1.1, so the before/after difference is attributable to the
pipeline order alone.

## Positive-class convention

**`Attack` (label 1) is the positive class** — `TRD.md §2.3`, the opposite of the base paper's
choice. Every metric below is hand-verified from the raw confusion matrix under this convention.
This is the check that would have caught the base paper's swapped precision/recall labels
(Objection #3, `PRD.md §2.1.3` / `§10`) before publication, and `TRD.md §2.3` requires we apply it
to our own results, not just theirs.

In [1]:
import logging
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)

from src.config import RANDOM_STATE
from src.evaluation.leakage_check import (
    check_leakage,
    compare_reports,
    synthetic_contamination,
)
from src.evaluation.metrics import (
    POSITIVE_CLASS_STATEMENT,
    compute_metrics,
    hand_verify_metrics,
    results_table,
)
from src.models.baseline import MAX_DEPTH, N_ESTIMATORS, fit_predict
from src.preprocessing.clean import clean_iotid20, load_iotid20
from src.preprocessing.feature_select import IOTID20_N_FEATURES, RandomForestFeatureSelector
from src.preprocessing.resample import (
    resample_training_fold,
    scale_split,
    split_dataset,
)

print(POSITIVE_CLASS_STATEMENT)
print(f"\nSeed: {RANDOM_STATE} | baseline RF: {N_ESTIMATORS} trees, max_depth={MAX_DEPTH} "
      "(identical to T1.1)")

POSITIVE CLASS = Attack (label 1); negative class = Normal (label 0). This is TRD.md §2.3's convention and is the OPPOSITE of the base paper's, which uses Normal as positive (PRD.md §2.1.3).

Seed: 42 | baseline RF: 100 trees, max_depth=20 (identical to T1.1)


## Step 1 — Load and clean (identical to T1.1)

In [2]:
raw = load_iotid20()
X, y, meta = clean_iotid20(raw, drop_duplicates=False)
print(f"Cleaned: {X.shape[0]:,} rows × {X.shape[1]} features")
print(f"Attack (positive, 1): {int((y == 1).sum()):,}   Normal (0): {int((y == 0).sum()):,}")

INFO | Loaded IoTID20: 625783 rows x 86 columns from /Users/rohithpranov/.cache/kagglehub/datasets/rohulaminlabid/iotid20-dataset/versions/2/IoT Network Intrusion Dataset.csv


INFO | Feature funnel: 83 raw -> 79 after removing 4 identifier columns


INFO | Replaced 736 infinite values with NaN


INFO | Dropped 368 rows containing NaN (625415 remain)


INFO | Dropped 10 zero-variance columns -> 69 features


INFO | Exact duplicate feature rows present: 363884 (drop_duplicates=False)


INFO | Clean complete: X=(625415, 69), class balance -> Attack(1)=585342, Normal(0)=40073


Cleaned: 625,415 rows × 69 features
Attack (positive, 1): 585,342   Normal (0): 40,073


## Step 2 — **Split first.** Nothing has been resampled or scaled yet.

80/10/10 train/test/validation, stratified on the label so every fold keeps IoTID20's real
93.6% / 6.4% class balance. The test and validation folds are now frozen: from here on, nothing
fitted on them can touch the model, and nothing synthetic can enter them.

In [3]:
split = split_dataset(X, y, random_state=RANDOM_STATE)
print(split.summary())

INFO | Split 625415 rows into 80/10/10 train/test/val
Pipeline order: HONEST : split BEFORE resample (TRD.md §2.2 -- T1.2)
  train       n=  500,332  Attack(1)=  468,274  Normal(0)=   32,058
  test        n=   62,541  Attack(1)=   58,534  Normal(0)=    4,007
  validation  n=   62,542  Attack(1)=   58,534  Normal(0)=    4,008


Pipeline order: HONEST : split BEFORE resample (TRD.md §2.2 -- T1.2)
  train       n=  500,332  Attack(1)=  468,274  Normal(0)=   32,058
  test        n=   62,541  Attack(1)=   58,534  Normal(0)=    4,007
  validation  n=   62,542  Attack(1)=   58,534  Normal(0)=    4,008


## Step 3 — Feature selection, fitted on the **training fold only** (83 → 62)

Same method and same 62-feature budget as T1.1 (Random Forest importance — *this project's own
choice*, not the base paper's unreproducible PSO; see `docs/feature_selection_decision.md`). The
difference is what it is allowed to see.

In [4]:
selector = RandomForestFeatureSelector(n_features=IOTID20_N_FEATURES, random_state=RANDOM_STATE)
selector.fit(split.X_train, split.y_train)
selector.ranking_.to_csv(REPO_ROOT / "reports" / "iotid20_feature_ranking_honest.csv", index=False)

split.X_train = selector.transform(split.X_train)
split.X_test = selector.transform(split.X_test)
split.X_val = selector.transform(split.X_val)
print(f"Selected {len(selector.selected_features_)} features on the training fold only")
selector.ranking_.head(15)

INFO | Fitting RF selector on (500332, 69) to rank 69 features


INFO | Selected 62/69 features; top 5: ['Dst_Port', 'Src_Port', 'Init_Bwd_Win_Byts', 'Flow_Duration', 'ACK_Flag_Cnt']


Selected 62 features on the training fold only


,feature,importance,rank,kept
0,Dst_Port,0.199602,1,True
1,Src_Port,0.131130,2,True
2,Init_Bwd_Win_Byts,0.106521,3,True
3,Flow_Duration,0.077196,4,True
4,ACK_Flag_Cnt,0.054442,5,True
5,Flow_Pkts/s,0.041322,6,True
6,Bwd_Header_Len,0.030975,7,True
7,Idle_Max,0.024237,8,True
8,Flow_IAT_Max,0.023763,9,True
9,Flow_IAT_Mean,0.021434,10,True


### How much did fitting the selector on the training fold change the choice?

If the two rankings agree closely, selection leakage was a minor effect on this dataset — worth
knowing, and worth reporting honestly either way.

In [5]:
leaky_ranking_path = REPO_ROOT / "reports" / "iotid20_feature_ranking_leaky.csv"
if leaky_ranking_path.exists():
    leaky_ranking = pd.read_csv(leaky_ranking_path)
    leaky_kept = set(leaky_ranking.loc[leaky_ranking["kept"], "feature"])
    honest_kept = set(selector.selected_features_)
    print(f"Features kept by both      : {len(leaky_kept & honest_kept)}")
    print(f"Only in the leaky ranking  : {sorted(leaky_kept - honest_kept)}")
    print(f"Only in the honest ranking : {sorted(honest_kept - leaky_kept)}")
else:
    print("Run 01_reproduce_leakage.ipynb first to enable this comparison.")

Features kept by both      : 61
Only in the leaky ranking  : ['Active_Min']
Only in the honest ranking : ['Active_Max']


## Step 4 — SMOTE, **training fold only**

`resample_training_fold` refuses to run on a split that came from the leaky path, so the broken
order cannot be reached by accident. Test and validation folds are returned untouched: they must
keep the real 93.6/6.4 class distribution, because that is the distribution the deployed system
would actually meet.

In [6]:
resampled = resample_training_fold(split, random_state=RANDOM_STATE)
print(resampled.summary())

INFO | SMOTE on TRAINING FOLD ONLY: 500332 -> 936548 rows (test/val untouched at 62541/62542)


Pipeline order: HONEST : split BEFORE resample (TRD.md §2.2 -- T1.2)
  train       n=  936,548  Attack(1)=  468,274  Normal(0)=  468,274
  test        n=   62,541  Attack(1)=   58,534  Normal(0)=    4,007
  validation  n=   62,542  Attack(1)=   58,534  Normal(0)=    4,008


## Step 5 — Scale, with the scaler fitted on the **training fold only**

In [7]:
honest = scale_split(resampled)
print(f"Scaler fitted on {len(honest.X_train):,} training rows, applied to all three folds")

INFO | MinMaxScaler fitted on the training fold (936548 rows) and applied to all folds


Scaler fitted on 936,548 training rows, applied to all three folds


## Step 6 — Confirm the test fold is clean

Zero synthetic rows, by construction. This is the mirror image of T1.1's Step 4.

In [8]:
honest_contamination = synthetic_contamination(
    honest.y_test, honest.synthetic_test, "HONEST (split before SMOTE)", fold_name="test"
)
print(honest_contamination.summary())

INFO | Synthetic-contamination check -- HONEST (split before SMOTE)
  test fold rows                 : 62,541
  of which SMOTE-generated              : 0 (0.00%)
  minority-class rows in the fold       : 4,007
  of those, SMOTE-generated             : 0.00%
  VERDICT: CLEAN: every row of the test fold is real observed traffic.


Synthetic-contamination check -- HONEST (split before SMOTE)
  test fold rows                 : 62,541
  of which SMOTE-generated              : 0 (0.00%)
  minority-class rows in the fold       : 4,007
  of those, SMOTE-generated             : 0.00%
  VERDICT: CLEAN: every row of the test fold is real observed traffic.


## Step 7 — Train and evaluate

**Positive class = Attack (label 1)**, per `TRD.md §2.3`.

In [9]:
model, y_pred = fit_predict(
    honest.X_train, honest.y_train, honest.X_test, random_state=RANDOM_STATE
)
honest_metrics = compute_metrics(honest.y_test, y_pred, "HONEST baseline (split before SMOTE)")
print(honest_metrics.report())

INFO | Fitting baseline RF on ((936548, 62),)


HONEST baseline (split before SMOTE)
  POSITIVE CLASS = Attack (label 1); negative class = Normal (label 0). This is TRD.md §2.3's convention and is the OPPOSITE of the base paper's, which uses Normal as positive (PRD.md §2.1.3).
  Confusion matrix (Attack = positive):
      TP (Attack  -> Attack) = 58,510
      FN (Attack  -> Normal) = 24   <- missed attacks
      FP (Normal  -> Attack) = 51   <- false alarms
      TN (Normal  -> Normal) = 3,956
  Accuracy  = (TP+TN)/(TP+FP+TN+FN) = 0.998801
  Precision = TP/(TP+FP)            = 0.999129
  Recall    = TP/(TP+FN)            = 0.999590
  F1        = 2PR/(P+R)             = 0.999359
  hand-verified against the confusion matrix: False


### Hand-verification against the confusion matrix

`TRD.md §2.3` requires this before any table is published. Each metric is recomputed from the raw
TP/FP/TN/FN cells using the literal formulas, then asserted equal to scikit-learn's value. An
`AssertionError` here would mean the positive-class convention had drifted somewhere in our own
code — Objection #3 happening to us.

In [10]:
honest_metrics = hand_verify_metrics(honest_metrics, verbose=True)

Hand-verification of: HONEST baseline (split before SMOTE)
  POSITIVE CLASS = Attack (label 1); negative class = Normal (label 0). This is TRD.md §2.3's convention and is the OPPOSITE of the base paper's, which uses Normal as positive (PRD.md §2.1.3).
  Accuracy  = (58,510 + 3,956) / 62,541 = 0.998800787
  Precision = 58,510 / (58,510 + 51) = 0.999129113
  Recall    = 58,510 / (58,510 + 24) = 0.999589982
  F1        = 2*0.999129*0.999590 / (0.999129+0.999590) = 0.999359494
    OK accuracy: hand 0.998800787 == sklearn 0.998800787
    OK precision: hand 0.999129113 == sklearn 0.999129113
    OK recall: hand 0.999589982 == sklearn 0.999589982
    OK f1: hand 0.999359494 == sklearn 0.999359494


In [11]:
# Independent cross-check: recompute with the roles reversed to show the labels are NOT swapped.
# With Normal as positive (the BASE PAPER'S convention) the numbers change — which is precisely
# why a results table is meaningless without stating its convention.
tp, fp, tn, fn = honest_metrics.tp, honest_metrics.fp, honest_metrics.tn, honest_metrics.fn
print("Same confusion matrix, both conventions:\n")
print(f"{'':<28}{'Attack positive (OURS)':>24}{'Normal positive (paper)':>26}")
print(f"{'Precision':<28}{tp / (tp + fp):>24.6f}{tn / (tn + fn):>26.6f}")
print(f"{'Recall':<28}{tp / (tp + fn):>24.6f}{tn / (tn + fp):>26.6f}")
print("\nThe two columns differ. Any table omitting its convention is ambiguous — that ambiguity")
print("is what Objection #3 is about, and it is why every table in this repo states its own.")

Same confusion matrix, both conventions:

                              Attack positive (OURS)   Normal positive (paper)
Precision                                   0.999129                  0.993970
Recall                                      0.999590                  0.987272

The two columns differ. Any table omitting its convention is ambiguous — that ambiguity
is what Objection #3 is about, and it is why every table in this repo states its own.


## Step 8 — Validation fold

The validation fold was never used for fitting or selection. Close agreement with the test fold
indicates the test number is not itself a lucky draw.

In [12]:
y_pred_val = model.predict(honest.X_val)
val_metrics = hand_verify_metrics(
    compute_metrics(honest.y_val, y_pred_val, "HONEST baseline (validation fold)"), verbose=False
)
print(val_metrics.report())

HONEST baseline (validation fold)
  POSITIVE CLASS = Attack (label 1); negative class = Normal (label 0). This is TRD.md §2.3's convention and is the OPPOSITE of the base paper's, which uses Normal as positive (PRD.md §2.1.3).
  Confusion matrix (Attack = positive):
      TP (Attack  -> Attack) = 58,508
      FN (Attack  -> Normal) = 26   <- missed attacks
      FP (Normal  -> Attack) = 63   <- false alarms
      TN (Normal  -> Normal) = 3,945
  Accuracy  = (TP+TN)/(TP+FP+TN+FN) = 0.998577
  Precision = TP/(TP+FP)            = 0.998924
  Recall    = TP/(TP+FN)            = 0.999556
  F1        = 2PR/(P+R)             = 0.999240
  hand-verified against the confusion matrix: True


## Step 9 — Nearest-neighbour check on the honest pipeline

Compared side by side with T1.1's. Read this table with the caveat stated in notebook 01: IoTID20
has 363,884 exact duplicate rows in the 69-feature space (58.2% of it), so the honest pipeline
shows some near-zero distances too.
The *direct* evidence is the synthetic-contamination contrast — T1.1's test fold is substantially
fabricated, this one is 0.00%.

In [13]:
honest_nn = check_leakage(
    honest.X_train, honest.y_train, honest.X_test, honest.y_test,
    pipeline_name="HONEST (split before SMOTE)",
)
print(honest_nn.summary())

INFO | Minority class of the training fold: 0


INFO | Nearest-neighbour leakage check -- HONEST (split before SMOTE)
  test samples examined (class 0) : 4,007 of 4,007
  training samples searched                        : 936,548
  distances < 1e-06                          : 1,690 (42.18%)
  distance percentiles:
      p0   : 0
      p1   : 0
      p5   : 0
      p25  : 0
      p50  : 1.95169e-05
      p75  : 8.80829e-05
      p100 : 1.08552
  VERDICT: LEAKAGE DETECTED


Nearest-neighbour leakage check -- HONEST (split before SMOTE)
  test samples examined (class 0) : 4,007 of 4,007
  training samples searched                        : 936,548
  distances < 1e-06                          : 1,690 (42.18%)
  distance percentiles:
      p0   : 0
      p1   : 0
      p5   : 0
      p25  : 0
      p50  : 1.95169e-05
      p75  : 8.80829e-05
      p100 : 1.08552
  VERDICT: LEAKAGE DETECTED


## Step 10 — Before / after

The headline of Contribution 1.

In [14]:
leaky_path = REPO_ROOT / "reports" / "t1_1_leaky_result.csv"
honest_summary = results_table([honest_metrics, val_metrics])
honest_summary["pipeline_order"] = "split BEFORE resample (TRD.md §2.2)"
honest_summary["test_fold_synthetic_fraction"] = [honest_contamination.fraction_synthetic, np.nan]
honest_summary["nn_near_zero_fraction"] = [honest_nn.near_zero_fraction, np.nan]

if leaky_path.exists():
    comparison = pd.concat([pd.read_csv(leaky_path), honest_summary], ignore_index=True)
else:
    comparison = honest_summary

comparison.to_csv(REPO_ROOT / "reports" / "t1_2_baseline_comparison.csv", index=False)
comparison.T

,0,1,2
model,LEAKY pipeline (SMOTE before split),HONEST baseline (split before SMOTE),HONEST baseline (validation fold)
positive_class,Attack,Attack,Attack
accuracy,0.998556,0.998801,0.998577
precision,0.997494,0.999129,0.998924
recall,0.999624,0.99959,0.999556
f1,0.998558,0.999359,0.99924
TP,58512,58510,58508
FP,147,51,63
TN,58387,3956,3945
FN,22,24,26


In [15]:
if leaky_path.exists():
    leaky_row = pd.read_csv(leaky_path).iloc[0]
    print(POSITIVE_CLASS_STATEMENT)
    print("\n" + "=" * 78)
    print(f"{'':<14}{'LEAKY (T1.1)':>16}{'HONEST (T1.2)':>18}{'change':>16}")
    print("=" * 78)
    for metric in ("accuracy", "precision", "recall", "f1"):
        lo, ho = float(leaky_row[metric]), getattr(honest_metrics, metric)
        print(f"{metric:<14}{lo:>16.6f}{ho:>18.6f}{ho - lo:>+16.6f}")
    print("-" * 78)
    print(f"{'test fold':<14}{leaky_row['test_fold_synthetic_fraction']:>15.2%} synthetic"
          f"{honest_contamination.fraction_synthetic:>17.2%} synthetic")
    print("=" * 78)

POSITIVE CLASS = Attack (label 1); negative class = Normal (label 0). This is TRD.md §2.3's convention and is the OPPOSITE of the base paper's, which uses Normal as positive (PRD.md §2.1.3).

                  LEAKY (T1.1)     HONEST (T1.2)          change
accuracy              0.998556          0.998801       +0.000244
precision             0.997494          0.999129       +0.001635
recall                0.999624          0.999590       -0.000034
f1                    0.998558          0.999359       +0.000802
------------------------------------------------------------------------------
test fold              46.61% synthetic            0.00% synthetic


## Step 11 — The result that did not go as expected, and what it means

**T1.2's VERIFY block predicts the honest accuracy will be *measurably lower* than T1.1's. On
IoTID20 it is not — it is marginally higher.** That is not a bug to tune away; it is a finding,
and burying it would be precisely the kind of convenient omission this project criticises.

Two things explain it, and the second is the important one:

1. **The two test folds are different populations.** T1.1's fold is ~50/50 Attack/Normal (a slice
   of the balanced post-SMOTE dataset); this one keeps IoTID20's real 93.6/6.4 imbalance, where
   accuracy is dominated by an easy majority class. The two accuracies are not commensurable.
2. **A second, larger leakage vector survives the fix.** Step 9's nearest-neighbour check on the
   *honest* pipeline still shows ~42% of minority test rows sitting at near-zero distance from a
   training row. Those are not synthetic — Step 6 proved the fold is 0.00% synthetic. They are
   **exact duplicate records**: 363,884 of IoTID20's 625,415 cleaned rows (58.2%) are duplicates
   in the 69-feature space. Random splitting therefore puts identical records on both sides
   regardless of when SMOTE runs.

So: fixing the SMOTE ordering removes *fabricated* test data (46.61% → 0.00%), which is real and
necessary. But on this dataset it is not the binding constraint on how trustworthy the number is.
**Deduplication is.** Neither the base paper nor the senior's prior work mentions it.

The run below applies both fixes. This is the number this project cites as its baseline.

In [16]:
X_dedup, y_dedup, _ = clean_iotid20(raw, drop_duplicates=True)
print(f"Deduplicated: {X_dedup.shape[0]:,} rows (from {X.shape[0]:,}) × {X_dedup.shape[1]} features")
print(f"Attack (positive, 1): {int((y_dedup == 1).sum()):,}   Normal (0): {int((y_dedup == 0).sum()):,}")
print(f"Class imbalance: {(y_dedup == 1).mean():.1%} Attack")

INFO | Feature funnel: 83 raw -> 79 after removing 4 identifier columns


INFO | Replaced 736 infinite values with NaN


INFO | Dropped 368 rows containing NaN (625415 remain)


INFO | Dropped 10 zero-variance columns -> 69 features


INFO | Exact duplicate feature rows present: 363884 (drop_duplicates=True)


INFO | Dropped duplicates -> 261531 rows remain


INFO | Clean complete: X=(261531, 69), class balance -> Attack(1)=234002, Normal(0)=27529


Deduplicated: 261,531 rows (from 625,415) × 69 features
Attack (positive, 1): 234,002   Normal (0): 27,529
Class imbalance: 89.5% Attack


In [17]:
split_d = split_dataset(X_dedup, y_dedup, random_state=RANDOM_STATE)

selector_d = RandomForestFeatureSelector(n_features=IOTID20_N_FEATURES, random_state=RANDOM_STATE)
selector_d.fit(split_d.X_train, split_d.y_train)
split_d.X_train = selector_d.transform(split_d.X_train)
split_d.X_test = selector_d.transform(split_d.X_test)
split_d.X_val = selector_d.transform(split_d.X_val)

dedup = scale_split(resample_training_fold(split_d, random_state=RANDOM_STATE))
print(dedup.summary())

INFO | Split 261531 rows into 80/10/10 train/test/val
Pipeline order: HONEST : split BEFORE resample (TRD.md §2.2 -- T1.2)
  train       n=  209,224  Attack(1)=  187,201  Normal(0)=   22,023
  test        n=   26,153  Attack(1)=   23,400  Normal(0)=    2,753
  validation  n=   26,154  Attack(1)=   23,401  Normal(0)=    2,753


INFO | Fitting RF selector on (209224, 69) to rank 69 features


INFO | Selected 62/69 features; top 5: ['Dst_Port', 'Flow_Duration', 'Src_Port', 'ACK_Flag_Cnt', 'Init_Bwd_Win_Byts']


INFO | SMOTE on TRAINING FOLD ONLY: 209224 -> 374402 rows (test/val untouched at 26153/26154)


INFO | MinMaxScaler fitted on the training fold (374402 rows) and applied to all folds


Pipeline order: HONEST : split BEFORE resample (TRD.md §2.2 -- T1.2)
  train       n=  374,402  Attack(1)=  187,201  Normal(0)=  187,201
  test        n=   26,153  Attack(1)=   23,400  Normal(0)=    2,753
  validation  n=   26,154  Attack(1)=   23,401  Normal(0)=    2,753


In [18]:
model_d, y_pred_d = fit_predict(
    dedup.X_train, dedup.y_train, dedup.X_test, random_state=RANDOM_STATE
)
dedup_metrics = hand_verify_metrics(
    compute_metrics(
        dedup.y_test, y_pred_d, "LEAKAGE-FREE baseline (split before SMOTE + deduplicated)"
    ),
    verbose=True,
)
print()
print(dedup_metrics.report())

INFO | Fitting baseline RF on ((374402, 62),)


Hand-verification of: LEAKAGE-FREE baseline (split before SMOTE + deduplicated)
  POSITIVE CLASS = Attack (label 1); negative class = Normal (label 0). This is TRD.md §2.3's convention and is the OPPOSITE of the base paper's, which uses Normal as positive (PRD.md §2.1.3).
  Accuracy  = (23,376 + 2,684) / 26,153 = 0.996444003
  Precision = 23,376 / (23,376 + 69) = 0.997056942
  Recall    = 23,376 / (23,376 + 24) = 0.998974359
  F1        = 2*0.997057*0.998974 / (0.997057+0.998974) = 0.998014729
    OK accuracy: hand 0.996444003 == sklearn 0.996444003
    OK precision: hand 0.997056942 == sklearn 0.997056942
    OK recall: hand 0.998974359 == sklearn 0.998974359
    OK f1: hand 0.998014729 == sklearn 0.998014729

LEAKAGE-FREE baseline (split before SMOTE + deduplicated)
  POSITIVE CLASS = Attack (label 1); negative class = Normal (label 0). This is TRD.md §2.3's convention and is the OPPOSITE of the base paper's, which uses Normal as positive (PRD.md §2.1.3).
  Confusion matrix (Attack =

In [19]:
dedup_contamination = synthetic_contamination(
    dedup.y_test, dedup.synthetic_test, "DEDUPLICATED honest", fold_name="test"
)
dedup_nn = check_leakage(
    dedup.X_train, dedup.y_train, dedup.X_test, dedup.y_test,
    pipeline_name="DEDUPLICATED honest (split before SMOTE, duplicates removed)",
)
print(dedup_contamination.summary())
print()
print(dedup_nn.summary())

INFO | Synthetic-contamination check -- DEDUPLICATED honest
  test fold rows                 : 26,153
  of which SMOTE-generated              : 0 (0.00%)
  minority-class rows in the fold       : 2,753
  of those, SMOTE-generated             : 0.00%
  VERDICT: CLEAN: every row of the test fold is real observed traffic.


INFO | Minority class of the training fold: 0


INFO | Nearest-neighbour leakage check -- DEDUPLICATED honest (split before SMOTE, duplicates removed)
  test samples examined (class 0) : 2,753 of 2,753
  training samples searched                        : 374,402
  distances < 1e-06                          : 157 (5.70%)
  distance percentiles:
      p0   : 0
      p1   : 0
      p5   : 2.77586e-07
      p25  : 2.39808e-05
      p50  : 5.50373e-05
      p75  : 0.000187487
      p100 : 0.42472
  VERDICT: LEAKAGE DETECTED


Synthetic-contamination check -- DEDUPLICATED honest
  test fold rows                 : 26,153
  of which SMOTE-generated              : 0 (0.00%)
  minority-class rows in the fold       : 2,753
  of those, SMOTE-generated             : 0.00%
  VERDICT: CLEAN: every row of the test fold is real observed traffic.

Nearest-neighbour leakage check -- DEDUPLICATED honest (split before SMOTE, duplicates removed)
  test samples examined (class 0) : 2,753 of 2,753
  training samples searched                        : 374,402
  distances < 1e-06                          : 157 (5.70%)
  distance percentiles:
      p0   : 0
      p1   : 0
      p5   : 2.77586e-07
      p25  : 2.39808e-05
      p50  : 5.50373e-05
      p75  : 0.000187487
      p100 : 0.42472
  VERDICT: LEAKAGE DETECTED


### The three pipelines side by side

**Positive class = Attack (label 1)** in every column (`TRD.md §2.3`).

In [20]:
leaky_row = pd.read_csv(leaky_path).iloc[0] if leaky_path.exists() else None

rows = [
    ("A. LEAKY (base paper's order)", leaky_row,
     float(leaky_row["nn_near_zero_fraction"]) if leaky_row is not None else np.nan),
    ("B. Honest order, duplicates kept", honest_metrics, honest_nn.near_zero_fraction),
    ("C. Honest order + deduplicated", dedup_metrics, dedup_nn.near_zero_fraction),
]

print(POSITIVE_CLASS_STATEMENT)
print("\n" + "=" * 100)
print(f"{'pipeline':<36}{'accuracy':>11}{'precision':>11}{'recall':>10}{'F1':>10}"
      f"{'synthetic':>12}{'NN near-0':>11}")
print("=" * 100)
for name, m, nn_frac in rows:
    if m is None:
        continue
    get = (lambda k: float(m[k])) if isinstance(m, pd.Series) else (lambda k: getattr(m, k))
    synth = (
        float(m["test_fold_synthetic_fraction"]) if isinstance(m, pd.Series)
        else (dedup_contamination.fraction_synthetic if "dedup" in name.lower()
              else honest_contamination.fraction_synthetic)
    )
    print(f"{name:<36}{get('accuracy'):>11.6f}{get('precision'):>11.6f}"
          f"{get('recall'):>10.6f}{get('f1'):>10.6f}{synth:>11.2%}{nn_frac:>11.2%}")
print("=" * 100)
print("\n'synthetic' = share of the test fold fabricated by SMOTE.")
print("'NN near-0' = share of minority test rows at <1e-6 distance from a training row.")
print("\nC is this project\'s baseline. Every later result is compared against C, not A or B.")

POSITIVE CLASS = Attack (label 1); negative class = Normal (label 0). This is TRD.md §2.3's convention and is the OPPOSITE of the base paper's, which uses Normal as positive (PRD.md §2.1.3).

pipeline                               accuracy  precision    recall        F1   synthetic  NN near-0
A. LEAKY (base paper's order)          0.998556   0.997494  0.999624  0.998558     46.61%     43.98%
B. Honest order, duplicates kept       0.998801   0.999129  0.999590  0.999359      0.00%     42.18%
C. Honest order + deduplicated         0.996444   0.997057  0.998974  0.998015      0.00%      5.70%

'synthetic' = share of the test fold fabricated by SMOTE.
'NN near-0' = share of minority test rows at <1e-6 distance from a training row.

C is this project's baseline. Every later result is compared against C, not A or B.


In [21]:
final = results_table([dedup_metrics])
final["pipeline_order"] = "split BEFORE resample + deduplicated (TRD.md §2.2 + T1.2 Step 11)"
final["test_fold_synthetic_fraction"] = dedup_contamination.fraction_synthetic
final["nn_near_zero_fraction"] = dedup_nn.near_zero_fraction
final.to_csv(REPO_ROOT / "reports" / "t1_2_leakage_free_baseline.csv", index=False)
final.T

,0
model,LEAKAGE-FREE baseline (split before SMOTE + de...
positive_class,Attack
accuracy,0.996444
precision,0.997057
recall,0.998974
f1,0.998015
TP,23376
FP,69
TN,2684
FN,24


## Verdict (T1.2 VERIFY block)

The VERIFY condition for T1.2 is: *"Reported accuracy is measurably lower than T1.1's leaky
number; the notebook contains an explicit positive-class statement and a hand-verified metric."*

- **Positive-class statement:** stated in the header, printed by every metrics block, and carried
  as a column in every saved table.
- **Hand-verified metric:** Step 7 recomputes all four metrics from the raw confusion-matrix cells
  and asserts agreement with scikit-learn; Step 7's cross-check shows the numbers under the base
  paper's opposite convention, demonstrating why the label matters.
- **Lower, honest number:** **not observed as predicted, and reported as such.** See Step 11:
  removing the SMOTE ordering bug alone did *not* lower the accuracy on IoTID20, because a larger
  leakage vector (58.2% duplicate rows) survives it. Applying both fixes gives pipeline C, which
  is the number this project cites. The VERIFY block's expectation was reasonable but turned out
  to be wrong on this dataset; `Build-Instructions.md` T1.2 should be annotated accordingly.


### What this baseline is, and is not

This is a **Random Forest**, not the project's architecture. It exists to establish an honest
reference point on a leakage-free pipeline. The committed architecture — the CNN + BiLSTM +
Transformer ensemble over genuine flow *sequences* (`TRD.md §3`) — is built in Phase 2 and
evaluated against this exact number in `reports/phase2_results.md`.

### On comparing any of this to the base paper

HIDS-IoMT's published 99.92% / 99.91% / 99.99% / 99.95% must never be cited in this project
without noting, **in the same paragraph**, that its precision and recall labels are very likely
swapped (`PRD.md §2.1.3`, worked in full at `PRD.md §10`) and that its evaluation fold was
contaminated in the manner reproduced in notebook 01. Those numbers are the object of a critique,
not a target to approach.